## Time to get your hands dirty. Your first neural network; pick your favourite.

*(exam guidelines available [here](https://github.com/dgerosa/astrostatistics_bicocca_2026/blob/main/exams.md))*


For the last coding assignment, you'll need to implement a neural network. We'll look at a relatively simple binary classification problem. Here below are three options; completing one of them for the exam is enough

### Tasks:

1. Remember: scale your data appropriately

2. Decide on a testing strategy (a simple test/train split? a CV strategy? set a test set aside to be looked at at the very end?)

2. Decide your optimization metric.

3. Write down your network architecture. You can start from a fully connected, multi-layer perceptron (and then explore)

4. Use one the package among those we've seen. These include Tensorflow via keras, pytorch, and the MPL classifier implemented in scikit-learn. This is an opportunity to pick the one you're most interested in learning. 

5. Optimize the hyperparameters of your network. Explore different hyperparameters and see what fits the data best.  Do your best now to optimize the network architecture. Be creative!

6. Report on the perfomance of the network on the test set; report other metrics that have not been optimized.


### A few tips:

- In scikit-learn, remember that you can utilize all availables cores on your machine with `n_jobs=-1`. Print out the classification score for the training data, and the best parameters obtained by the cross validation.
- If it takes too long, run the hyperparameter optimization on a subset of the training set. Then retrain the full network using the best hyperparameters only.
- On cross validation, for scikit learn we've seen how to use `GridSearchCV` already. For Tensorflow, there's a really cool tool called [Tensorboard](https://www.tensorflow.org/tensorboard)

### Datasets:

You can choose one of these three problems:

- **1. Galaxies vs quasars (but with neural networks)** Go back to our SDSS data we've used in Lecture 19. We had color differences, and the task was to classifty quasars vs galaxies. Repeat that task with a neural network.

- **2. Can a computer learn if we're going to detect gravitational waves? (but with neural networks)** Go back to the SNR classifier for gravitational wave events, same data we've used in Lecture. We had properties of black hole binaries, and the task was to classify. Repeat that task with a neural network.

- **3. The HiggsML challenge** Branching out of astrophysics, let's mess around with a dataset of simulated but  realistic events from the ATLAS particle detector at CERN.
    - Data are at `solutions/higgs.tar.gz` (you need to uncompress with `tar -czvf`)
    - There are $N_{\rm samples} = 2.5\times 10^5$ entries with $N_{\rm features}=30$ features each. 
    - The taks is that of classifying these features against a set of labels, which are either `s` (source) or `b` (background).
    - For some info on both the physics and the dataset see [this document](https://higgsml.lal.in2p3.fr/files/2014/04/documentation_v1.8.pdf); includes a description of the features and how data have been padded (-999) for missing values.
    - This dataset was part of a challenge that run on Kaggle in 2014: https://higgsml.ijclab.in2p3.fr/ 




In [1]:
import pandas as pd 
import tensorflow
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

I0000 00:00:1782482481.055683   17508 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782482481.056375   17508 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1782482481.171672   17508 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782482483.081277   17508 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
df = pd.read_csv('/home/matti/uni/astrostatistics_bicocca_2026/solutions/higgs.csv')


In [3]:
df_filt = df.loc[:, ~df.columns.isin(['Label', 'KaggleSet', 'KaggleWeight'])]

In [4]:
y = df['Label']

In [5]:
y = np.array([1 if label == 's' else 0 for label in y], dtype='int32')

In [6]:
y

array([1, 0, 0, ..., 1, 0, 0], shape=(250000,), dtype=int32)

In [7]:
df_filt

,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,258.733,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,164.546,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,260.414,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,86.062,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,53.131,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,349995,-999.000,71.989,36.548,5.042,-999.00,-999.000,-999.000,1.392,5.042,...,144.665,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,4.505083
249996,349996,-999.000,58.179,68.083,22.439,-999.00,-999.000,-999.000,2.585,22.439,...,80.408,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,2.497259
249997,349997,105.457,60.526,75.839,39.757,-999.00,-999.000,-999.000,2.390,22.183,...,198.907,1,41.992,1.800,-0.166,-999.000,-999.00,-999.000,41.992,0.018636
249998,349998,94.951,19.362,68.812,13.504,-999.00,-999.000,-999.000,3.365,13.504,...,112.718,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,1.681611


In [8]:
df_filt.columns

Index(['EventId', 'DER_mass_MMC', 'DER_mass_transverse_met_lep',
       'DER_mass_vis', 'DER_pt_h', 'DER_deltaeta_jet_jet', 'DER_mass_jet_jet',
       'DER_prodeta_jet_jet', 'DER_deltar_tau_lep', 'DER_pt_tot', 'DER_sum_pt',
       'DER_pt_ratio_lep_tau', 'DER_met_phi_centrality',
       'DER_lep_eta_centrality', 'PRI_tau_pt', 'PRI_tau_eta', 'PRI_tau_phi',
       'PRI_lep_pt', 'PRI_lep_eta', 'PRI_lep_phi', 'PRI_met', 'PRI_met_phi',
       'PRI_met_sumet', 'PRI_jet_num', 'PRI_jet_leading_pt',
       'PRI_jet_leading_eta', 'PRI_jet_leading_phi', 'PRI_jet_subleading_pt',
       'PRI_jet_subleading_eta', 'PRI_jet_subleading_phi', 'PRI_jet_all_pt',
       'Weight'],
      dtype='object')

In [9]:
df_filt_nan = df_filt.replace(-999, np.nan)

In [10]:
print(df_filt_nan.isna().sum())  # NaN count per column
print(df_filt_nan.isna().sum().sum())  # Total NaN count
print(len(df_filt_nan.dropna()))  # Rows remaining after dropna()

EventId                             0
DER_mass_MMC                    38114
DER_mass_transverse_met_lep         0
DER_mass_vis                        0
DER_pt_h                            0
DER_deltaeta_jet_jet           177457
DER_mass_jet_jet               177457
DER_prodeta_jet_jet            177457
DER_deltar_tau_lep                  0
DER_pt_tot                          0
DER_sum_pt                          0
DER_pt_ratio_lep_tau                0
DER_met_phi_centrality              0
DER_lep_eta_centrality         177457
PRI_tau_pt                          0
PRI_tau_eta                         0
PRI_tau_phi                         0
PRI_lep_pt                          0
PRI_lep_eta                         0
PRI_lep_phi                         0
PRI_met                             0
PRI_met_phi                         0
PRI_met_sumet                       0
PRI_jet_num                         0
PRI_jet_leading_pt              99913
PRI_jet_leading_eta             99913
PRI_jet_lead

In [11]:
scaler = StandardScaler()
scaled = scaler.fit_transform(df_filt_nan.values)

In [12]:
scaled

array([[-1.73204388,  0.28991353,  0.06833197, ..., -1.36131161,
         0.4125105 , -0.87681426],
       [-1.73203002,  0.68202131,  0.55250482, ...,         nan,
        -0.27381996,  0.31295252],
       [-1.73201617,         nan,  3.19515553, ...,         nan,
        -0.29396985,  0.37364501],
       ...,
       [ 1.73201617, -0.28624947,  0.31931645, ...,         nan,
        -0.31701723, -0.86829055],
       [ 1.73203002, -0.46960659, -0.84532397, ...,         nan,
        -0.74543941,  0.01858253],
       [ 1.73204388,         nan,  0.66533608, ...,         nan,
        -0.74543941,  0.12303692]], shape=(250000, 32))

I try different models with different number of layers/fuctions/neurons in a cv scheme

In [13]:
from sklearn.model_selection import train_test_split, KFold
from keras import layers

In [14]:
keras.backend.clear_session()
#model1 = 2 layer, all relu with differrent dropouts to prevent overfitting
model1 = keras.Sequential([
    layers.Input(shape = (32,)),
    layers.Dense(32, activation='relu', ),#kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(32, activation='relu', ),#kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])
# model1.summary()
model2 = keras.Sequential([
    layers.Input(shape = (32, )),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
# model2.summary()
model3 = keras.Sequential([
    layers.Input(shape = (32,)),
    layers.Dense(64, activation='tanh'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='tanh'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
# model3.summary()
model4 = keras.Sequential([
    layers.Input(shape = (32, )),
    layers.Dense(64, activation='tanh'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='tanh'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='tanh'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='tanh'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
# model4.summary()

E0000 00:00:1782482486.663036   17508 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [15]:
data_train, data_test, y_train, y_test = train_test_split(scaled, y, test_size= 0.15)
# prova1, prova2, y1, y2 = train_test_split(data_train, y_train, test_size= 0.2)

In [16]:
y_train

array([0, 0, 0, ..., 0, 1, 0], shape=(212500,), dtype=int32)

In [21]:
kfold = KFold()

for (train_index, cv_test_index) in kfold.split(data_train):
    keras.backend.clear_session()
    data_cv_train, data_cv_test = data_train[train_index], data_train[cv_test_index]
    y_cv_train, y_cv_test = y_train[train_index], y_train[cv_test_index]
    model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model1.fit(data_cv_train, y_cv_train, epochs = 5)
    model1.evaluate(data_cv_test, y_cv_test)
    loss, accuracy = model1.evaluate(data_cv_test, y_cv_test)
    print(f"Accuracy: {accuracy}")


Epoch 1/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.6565 - loss: 0.6434
Epoch 2/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.6565 - loss: 0.6435
Epoch 3/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.6565 - loss: 0.6434
Epoch 4/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.6565 - loss: 0.6434
Epoch 5/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.6565 - loss: 0.6434
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6575 - loss: 0.6427
1329/1329 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6575 - loss: 0.6427
Accuracy: 0.6575294137001038
Epoch 1/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.6564 - loss: 0.6435
Epoch 2/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.6564 - loss: 0.6435
Epoch 3/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.6564 - loss: 0.6434
Epoch 4/5
5313/5313 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.6564 - loss: 0.6435
Epoch 5/5
5313/5313 ━━━━━

In [ ]:
kfold = KFold()
models = [model1, model2, model3, model4]
acc = []
acc_std = []
for model in models:
    acc_cv = []
    for (train_index, cv_test_index) in kfold.split(data_train):
        keras.backend.clear_session()
        data_cv_train, data_cv_test = data_train[train_index], data_train[cv_test_index]
        y_cv_train, y_cv_test = y_train[train_index], y_train[cv_test_index]
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        model.fit(data_cv_train, y_cv_train, epochs = 5)
        model.evaluate(data_cv_test, y_cv_test)
        loss, accuracy = model.evaluate(data_cv_test, y_cv_test)
        acc_cv.append(accuracy)

    acc.append(np.mean(acc_cv))
    acc_std.append(np.std(acc_cv))